In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_openai import ChatOpenAI

/Users/ankitdhandharia/Documents/Projects/RAG_Project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
llm = ChatOpenAI(
    model="liquid/lfm-2.5-1.2b-thinking:free",
    openai_api_key=os.getenv("OPENROUTER_API_KEY"),
    openai_api_base="https://openrouter.ai/api/v1",
)

In [5]:
response = llm.invoke("What is the meaning of life in one line?")
print(response.content)

## **RAG IMPLEMENTATION WITH OUR OWN TEXT DATA**

### **STEP 1: PREPARNG DOCUMENT From PDFs**

In [6]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path = "./docs/content2.pdf"
loader = PyPDFLoader(pdf_path)
docs = loader.load()
docs

[Document(metadata={'producer': 'Acrobat Distiller 15.0 (Windows)', 'creator': 'ACOMP.exe   WinVer 1c15  Aug 31 2005', 'creationdate': '2022-08-22T09:09:04-04:00', 'moddate': '2024-12-31T07:47:15-05:00', 'title': 'C:\\USERS\\ASABAL~1\\DESKTOP\\ORGANI~1\\CONST.18', 'source': './docs/content2.pdf', 'total_pages': 15, 'page': 0, 'page_label': 'i'}, page_content='Page I \n1 This text of the Constitution follows the engrossed copy \nsigned by Gen. Washington and the deputies from 12 States. The \nsmall superior figures preceding the paragraphs designate \nclauses, and were not in the original and have no reference to \nfootnotes. \nIn May 1785, a committee of Congress made a report rec-\nommending an alteration in the Articles of Confederation, but \nno action was taken on it, and it was left to the State Legisla-\ntures to proceed in the matter. In January 1786, the Legislature \nof Virginia passed a resolution providing for the appointment of \nfive commissioners, who, or any three of the

### **STEP 2: Splitting the data into chunks**

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap=100
)

chunks = splitter.split_documents(docs)
chunks

[Document(metadata={'producer': 'Acrobat Distiller 15.0 (Windows)', 'creator': 'ACOMP.exe   WinVer 1c15  Aug 31 2005', 'creationdate': '2022-08-22T09:09:04-04:00', 'moddate': '2024-12-31T07:47:15-05:00', 'title': 'C:\\USERS\\ASABAL~1\\DESKTOP\\ORGANI~1\\CONST.18', 'source': './docs/content2.pdf', 'total_pages': 15, 'page': 0, 'page_label': 'i'}, page_content='Page I \n1 This text of the Constitution follows the engrossed copy \nsigned by Gen. Washington and the deputies from 12 States. The \nsmall superior figures preceding the paragraphs designate \nclauses, and were not in the original and have no reference to \nfootnotes. \nIn May 1785, a committee of Congress made a report rec-\nommending an alteration in the Articles of Confederation, but \nno action was taken on it, and it was left to the State Legisla-\ntures to proceed in the matter. In January 1786, the Legislature \nof Virginia passed a resolution providing for the appointment of \nfive commissioners, who, or any three of the

In [8]:
len(chunks)

111

### **STEP 3: Creating Embeddings For The Chunks**

In [9]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"  
)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4132.04it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [10]:
embeddings = embedding_model.embed_query("What is the meaning of life?")
print(len(embeddings))  # 384 dimensions

384


### **STEP 4: Create and Store Embeddings in Exisiting Vector Store**

In [12]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma(
    embedding_function=embedding_model,
    persist_directory="./vector/"
)

In [13]:
vectorstore.add_documents(chunks)

['6385697e-2f8f-42dd-9c4f-9b162d28ca60',
 'e0c068aa-037d-4a57-a4bb-c8c4bbde95fb',
 'c5b5d3d3-b46f-4d7b-9fe9-39e303eae7d9',
 'bc6337af-0567-4367-a68d-68439a5069a6',
 'bc004aab-8355-41df-8f05-42861e4631b6',
 '2dfb33f1-ce54-42a7-9c29-ffe7a1b845bf',
 '4c359243-419c-4cad-88e9-bfa2ea1e414a',
 '67e38bfb-0709-4f94-afe1-9e429d71aade',
 '00f08726-5330-4a70-80d3-81e7ff448b45',
 '171807cb-43c1-4587-b252-5854a980c54a',
 'd992c2f4-bfa3-4c5f-b85f-1cd0b194bed7',
 'e1b8252c-d678-44a7-bd22-c7465240d3cf',
 '2f69cd18-fa8d-4c19-b7d9-712676543fa9',
 '179428ae-03ee-452e-a4a1-08349dee1306',
 '4d680d26-5d6c-4073-b89b-f0535f7395ee',
 '2f1a24ea-cc7e-4c4c-bbd8-1befe6593c15',
 '0d9639bb-a87c-438e-9643-384466f0eb4c',
 '664f2f8c-03b2-4fca-a5a4-98d04c2215d4',
 '194b3619-2926-4c7c-9154-b5a7ce6de488',
 'b4664ef3-62f2-48fc-926b-f0e640dc5627',
 '47349c6f-54d3-489e-a5cd-54ddc3b906af',
 '5267c11f-6012-4951-af77-7583fc4f4ea5',
 'f4a84b01-c705-452e-8d76-c4034c3f4941',
 'ea846c3b-5a0d-45cc-9bfe-0cddaf95b6a7',
 '8b7c4157-4e31-

### **STEP 5: Semantic Search**

In [17]:
question = "Who was elected President of the Constitutional Convention, and on what date did the convention begin?"
context = vectorstore.similarity_search(question, k=3)

In [18]:
context

[Document(metadata={'page': 0, 'creator': 'ACOMP.exe   WinVer 1c15  Aug 31 2005', 'moddate': '2024-12-31T07:47:15-05:00', 'creationdate': '2022-08-22T09:09:04-04:00', 'source': './docs/content2.pdf', 'total_pages': 15, 'page_label': 'i', 'title': 'C:\\USERS\\ASABAL~1\\DESKTOP\\ORGANI~1\\CONST.18', 'producer': 'Acrobat Distiller 15.0 (Windows)'}, page_content='promptly appointed delegates. On the 25th of May, seven States \nhaving convened, George Washington, of Virginia, was unani-\nmously elected President, and the consideration of the proposed \nconstitution was commenced. On the 17th of September, 1787, the \nConstitution as engrossed and agreed upon was signed by all the \nmembers present, except Mr. Gerry of Massachusetts, and \nMessrs. Mason and Randolph, of Virginia. The president of the \nconvention transmitted it to Congress, with a resolution stating \nhow the proposed Federal Government should be put in oper-\nation, and an explanatory letter. Congress, on the 28th of Sep-\n

### **Talk to LLM**

In [19]:
response = llm.invoke(f"{question} You can answer using the following context: {context}")
print(response.content)

The elected President of the Constitutional Convention was **George Washington**, and the convention began on **May 25, 1787**. 

Answer: The President was George Washington, and the convention started on May 25, 1787. 

\boxed{George Washington \text{ on May 25, 1787}}


In [20]:
print(response)

content='The elected President of the Constitutional Convention was **George Washington**, and the convention began on **May 25, 1787**. \n\nAnswer: The President was George Washington, and the convention started on May 25, 1787. \n\n\\boxed{George Washington \\text{ on May 25, 1787}}' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 781, 'prompt_tokens': 1301, 'total_tokens': 2082, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 816, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0, 'upstream_inference_prompt_cost': 0, 'upstream_inference_completions_cost': 0}}, 'model_provider': 'openai', 'model_name': 'liquid/lfm-2.5-1.2b-thinking-20260120:free', 'system_fingerprint': None, 'id': 'gen-1773948427-